# Python CUDA: NumPy, PyTorch, and CuPy

Kernel: **Python (CUDA Learning)**. CPU sections work without a GPU. GPU sections explicitly report a skip until NVIDIA CUDA becomes available.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import cupy as cp

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
torch_ready = torch.cuda.is_available()
try:
    cupy_ready = cp.cuda.runtime.getDeviceCount() > 0
except Exception as exc:
    cupy_ready = False
    print("CuPy GPU unavailable:", exc)
print("PyTorch GPU available:", torch_ready)
print("CuPy GPU available:", cupy_ready)
if torch_ready:
    print("GPU:", torch.cuda.get_device_name(0))

## 1. CPU arrays and tabular data

NumPy and pandas use CPU memory. CUDA execution requires an explicit transfer to a CUDA library.

In [ ]:
frame = pd.DataFrame({"quantity": [2, 5, 3], "price": [10.0, 8.0, 15.0]})
frame["revenue"] = frame["quantity"] * frame["price"]
a_cpu = np.arange(12, dtype=np.float32).reshape(3, 4)
expected = a_cpu @ a_cpu.T
print(frame)
print("\nCPU matrix product:\n", expected)

## 2. PyTorch on CUDA

`device='cuda'` copies an array to GPU memory. `.cpu()` brings a result back for NumPy validation.

In [ ]:
if torch_ready:
    a_gpu = torch.tensor(a_cpu, device="cuda")
    result = (a_gpu @ a_gpu.T).cpu().numpy()
    np.testing.assert_allclose(result, expected)
    print("PyTorch CUDA verified:", result)
else:
    print("SKIPPED: PyTorch CUDA requires a detected NVIDIA GPU.")

## 3. CuPy: NumPy-style GPU arrays

CuPy resembles NumPy, but its arrays live on the GPU. Use `cp.asarray` and `cp.asnumpy` for transfers.

In [ ]:
if cupy_ready:
    a_gpu = cp.asarray(a_cpu)
    result = cp.asnumpy(a_gpu @ a_gpu.T)
    np.testing.assert_allclose(result, expected)
    print("CuPy CUDA verified:", result)
else:
    print("SKIPPED: CuPy CUDA requires a detected NVIDIA GPU.")

## 4. Write a CUDA kernel from Python

Each GPU thread handles one array element. The bounds check protects the final partial block. CuPy compiles the kernel with NVRTC on first use.

In [ ]:
if cupy_ready:
    vector_add = cp.RawKernel(
        r"""
    extern "C" __global__
    void vector_add(const float* x, const float* y, float* out, int n) {
        int i = blockDim.x * blockIdx.x + threadIdx.x;
        if (i < n) out[i] = x[i] + y[i];
    }
    """,
        "vector_add",
    )
    n = 10003
    x = cp.arange(n, dtype=cp.float32)
    y = cp.ones(n, dtype=cp.float32)
    out = cp.empty_like(x)
    vector_add(((n + 255) // 256,), (256,), (x, y, out, np.int32(n)))
    np.testing.assert_allclose(cp.asnumpy(out), np.arange(n, dtype=np.float32) + 1)
    print("RawKernel vector addition passed.")
else:
    print("SKIPPED: custom kernel requires CUDA hardware.")

## 5. Measure GPU computation

CUDA work is asynchronous. Warm up first, then measure CUDA events and synchronize. These timings exclude host/device transfer and are not a CPU/GPU speedup comparison.

In [ ]:
if torch_ready:
    size = 1024
    x = torch.randn(size, size, device="cuda")
    for _ in range(3):
        result = x @ x.T
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(10):
        result = x @ x.T
    end.record()
    end.synchronize()
    print(f"{size} x {size} matmul: {start.elapsed_time(end) / 10:.3f} ms (mean)")
else:
    print("SKIPPED: no CUDA timings are available without a CUDA GPU.")

## Exercises

1. Change the CPU array shape and predict the matrix-product shape.
2. Transfer a pandas revenue column to CuPy and calculate its sum.
3. Change the kernel to `out[i] = 2 * x[i] + y[i]` and validate against NumPy.
4. Try matrix sizes 256, 512, and 2048. Keep synchronization in the timing code.